# 13 可重現研究 — 最小可驗證流程

用松柏護理之家退伍軍人症資料示範「從零到摘要」的可重現工作流程。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
# --- Step 1: 讀取資料並產出摘要 ---
path = Path("data/synthetic/legionella_outbreak.csv")
df = pd.read_csv(path)
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

summary = {
    "n_residents": len(df),
    "n_zones": df.groupby(["floor", "wing"]).ngroups,
    "n_infected": int(df["infected"].sum()),
    "n_deaths": int((df["outcome"] == "dead").sum()),
    "attack_rate": f"{df['infected'].mean():.1%}",
    "cfr": f"{(df['outcome'] == 'dead').sum() / df['infected'].sum():.1%}",
}

print("=== 疫情摘要 ===")
for k, v in summary.items():
    print(f"  {k}: {v}")

print("\n→ 這個 dict 就是最小可驗證結果")
print("→ 任何人在任何機器上跑都應該得到一樣的數字")

In [ ]:
# --- Step 2: 可重現檢查清單 ---
from pathlib import Path as _P

checks = {
    "uv.lock 存在": _P("uv.lock").exists(),
    "資料檔存在": _P("data/synthetic/legionella_outbreak.csv").exists(),
    "pyproject.toml 存在": _P("pyproject.toml").exists(),
    "tests/ 目錄存在": _P("tests").is_dir(),
}

print("=== 可重現檢查清單 ===")
for item, ok in checks.items():
    status = "✓" if ok else "✗"
    print(f"  [{status}] {item}")

all_pass = all(checks.values())
print(f"\n→ {'全部通過！環境可重現' if all_pass else '有項目未通過，需要修正'}")

In [ ]:
# --- Step 3: 將摘要寫入 CSV ---
import json

# 方法 1: 存成 CSV
summary_df = pd.DataFrame([summary])
output_path = Path("data/processed")
output_path.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(output_path / "summary.csv", index=False)

# 方法 2: 存成 JSON（保留型別）
with open(output_path / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("=== 輸出檔案 ===")
print(f"  CSV: {output_path / 'summary.csv'}")
print(f"  JSON: {output_path / 'summary.json'}")
print("\n→ 下次驗證時，比對這些檔案就知道結果是否一致")

In [ ]:
# --- Step 4: 版本資訊紀錄 ---
import sys
import platform

env_info = {
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
}

print("=== 環境版本資訊 ===")
for k, v in env_info.items():
    print(f"  {k}: {v}")

print("\n→ 將版本資訊附在報告中，他人才能重現你的環境")
print("→ 用 uv.lock 可以自動鎖定所有套件版本")

## 小結

| 步驟 | 學到的技能 |
|------|------------|
| 讀取 + 摘要 | 用 dict 產出最小可驗證結果 |
| 檢查清單 | 確認環境檔案齊全 |
| 輸出存檔 | CSV / JSON 保存結果供比對 |
| 版本資訊 | 記錄 Python / 套件版本 |

**可重現的三要素**：
1. **資料**：固定的輸入檔（`legionella_outbreak.csv`）
2. **程式碼**：版本控制（git commit）
3. **環境**：鎖定套件（`uv.lock`）

下一章（Ch14），我們把所有技能整合成一個完整實戰案例。